# Содержание
### Строим граф для 1 сбщ, при этом 1 раз валидируем промпт-экстрактор силами LLM
1. Стереть Neo4j базу до нуля (7688)
2. Вызов DeepSeek с промптом-экстрактором (cклеиваем строки в промпт (+ сбщ, + онто)). Выход - ABOX 
3. Полученный ABox, онто, сбщ передаем LLM и спрашиваем: соотв ли ABox всему? сделай в правки в промпт-экстр чтобы все было лучше. Сохраняем результат-промпт
4. Пункт 2 снова
5. ABox JSON -> Cypher конвертер (LLM возвращает JSON с узлами и нодами, чтобы в конт окно влезало. Cypher не влезал!)
6. Запись в файл cypher_checkpoints - слепки базы на каждое сбщ
7. Внесение инфы в базу (исполнение всех Cypher команд по очереди)

   

In [63]:
%pip -q install python-dotenv

from dotenv import load_dotenv
import os

load_dotenv(".env")   # подхватит /.../neo4j_граф/.env

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
print("DEEPSEEK_API_KEY loaded:", bool(DEEPSEEK_API_KEY))


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
DEEPSEEK_API_KEY loaded: True


In [140]:
# 1 стирание базы полное до нуля

In [163]:
def step_1_reset_db():
    !docker stop neo4j_travel || true
    !docker rm neo4j_travel || true
    !docker volume rm neo4j_travel_data || true
    
    !docker run -d --name neo4j_travel \
      -p 7475:7474 -p 7688:7687 \
      -e NEO4J_AUTH=neo4j/12345678 \
      -v neo4j_travel_data:/data \
      neo4j:5
    
    # !docker ps --filter name=neo4j_travel   
    print("Did reset DB")
    return None


In [166]:
# 2 сборка промпта и запрос к DeepSeek - выход ABox в виде JSON

In [167]:
import os
def step_2_prompt_to_abox(msg_as_JSON) -> str:
    import os
    import json
    import requests
    import time

    PROMPT_PATH = "prompt.txt"
    ONTO_PATH = "ontology.txt"

    DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
    if not DEEPSEEK_API_KEY:
        raise RuntimeError("Set env var DEEPSEEK_API_KEY (export DEEPSEEK_API_KEY=...).")

    DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL", "https://api.deepseek.com").rstrip("/")
    MODEL = os.getenv("DEEPSEEK_MODEL", "deepseek-reasoner")  # or deepseek-chat

    # --- load prompt template + ontology ---
    with open(PROMPT_PATH, "r", encoding="utf-8") as f:
        prompt_template = f.read()

    with open(ONTO_PATH, "r", encoding="utf-8") as f:
        ontology_text = f.read().strip()

    # --- normalize message JSON string ---
    if isinstance(msg_as_JSON, (dict, list)):
        message_json_string = json.dumps(msg_as_JSON, ensure_ascii=False)
    else:
        message_json_string = str(msg_as_JSON).strip()

    # validate that it's JSON (LLM expects JSON string with keys post_id/post_url/author/date/text)
    try:
        json.loads(message_json_string)
    except Exception as e:
        raise ValueError(f"msg_as_JSON must be a valid JSON string (or dict). Parse error: {e}") from e

    # --- stitch prompt ---
    prompt = (
        prompt_template
        .replace("{RDF_ONTOLOGY_HERE}", ontology_text)
        .replace("{MESSAGE_JSON_STRING_HERE}", message_json_string)
    )

    # sanity check for unresolved placeholders
    unresolved = [p for p in ["{RDF_ONTOLOGY_HERE}", "{MESSAGE_JSON_STRING_HERE}"] if p in prompt]
    if unresolved:
        raise RuntimeError(f"Unresolved placeholders in prompt: {unresolved}")

    # --- call DeepSeek ---
    url = f"{DEEPSEEK_BASE_URL}/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {DEEPSEEK_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": MODEL,
        "messages": [
            # system можно оставить очень коротким, промпт уже строгий
            {"role": "system", "content": "Return ONLY valid JSON object. No markdown. No code fences."},
            {"role": "user", "content": prompt},
        ],
        "temperature": 0.0,
    }

    t0 = time.perf_counter()
    resp = requests.post(url, headers=headers, json=payload, timeout=180)
    elapsed = time.perf_counter() - t0
    print(f"DeepSeek API latency: {elapsed:.2f}s")
    resp.raise_for_status()
    data = resp.json()
    print('Did recieve response from deepseek 1')
    api_finish = data["choices"][0].get("finish_reason", "")
    content = data["choices"][0]["message"]["content"]

    emoji = "🟢" if api_finish != "length" else "🔴"
    print(f"{emoji} API finish_reason = {api_finish}")

    # optional: validate JSON (hard fail if model returned junk)
    try:
        json.loads(content)
    except Exception as e:
        raise ValueError(f"Model did not return valid JSON. Error: {e}\nRaw content:\n{content}") from e

    print(content)  # preview
    return content


In [168]:
# print(answer)  # preview
# print(content)

In [169]:
# 3 исправление pr_ex силами LLM

In [170]:
def improve_prompt_with_llm(msg_as_json: str, abox_json: str) -> str:
    import os, json, time, requests

    PROMPT_PATH = "prompt.txt"
    ONTO_PATH = "ontology.txt"

    DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
    if not DEEPSEEK_API_KEY:
        raise RuntimeError("Set env var DEEPSEEK_API_KEY first: export DEEPSEEK_API_KEY=...")

    DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL", "https://api.deepseek.com").rstrip("/")
    MODEL = os.getenv("DEEPSEEK_MODEL", "deepseek-reasoner")  # or deepseek-reasoner

    with open(PROMPT_PATH, "r", encoding="utf-8") as f:
        current_prompt = f.read()

    with open(ONTO_PATH, "r", encoding="utf-8") as f:
        ontology_text = f.read().strip()

    meta_prompt = f"""
You are a prompt engineer for an ontology-guided ABox extractor.

INPUTS YOU WILL RECEIVE:
1) CURRENT_PROMPT: the current extraction prompt text that we use to ask an LLM to produce ABox JSON.
2) ONTOLOGY_TTL: the RDF ontology in Turtle format (authoritative).
3) MESSAGE_JSON: a single forum post as a JSON string with fields post_id (int), post_url (string), author (string), date (string), text (string).
4) EXTRACTED_ABOX_JSON: the last ABox JSON produced by the extractor for this message.

TASK
Produce an improved version of CURRENT_PROMPT so that when it is used again, the extractor LLM is more likely to:
- obey the ontology strictly,
- obey all key-generation rules,
- keep all string values in English ASCII (no Cyrillic),
- extract required metadata fields from MESSAGE_JSON

RULES
- Output ONLY the updated prompt text (no JSON, no markdown, no explanations).
- Keep the structure and placeholders used by the pipeline:
  - {{RDF_ONTOLOGY_HERE}} must remain as-is
  - {{MESSAGE_JSON_HERE}} must remain as-is
- Do NOT reference these instructions. Write the final prompt as a standalone extraction prompt.
- Fix contradictions and remove ambiguous wording.
- Incorporate “negative rules” derived from the observed errors.
- Do not change phrases RDF_ONTOLOGY_HERE and MESSAGE_JSON_STRING_HERE. They must appear exactly this way in the prompt

NOW UPDATE THE PROMPT.

CURRENT_PROMPT:
<<<CURRENT_PROMPT_START
{current_prompt}
CURRENT_PROMPT_END>>>

ONTOLOGY_TTL:
<<<ONTOLOGY_START
{ontology_text}
ONTOLOGY_END>>>

MESSAGE_JSON:
<<<MESSAGE_START
{msg_as_json}
MESSAGE_END>>>

EXTRACTED_ABOX_JSON:
<<<ABOX_START
{abox_json}
ABOX_END>>>
""".strip()

    url = f"{DEEPSEEK_BASE_URL}/v1/chat/completions"
    headers = {"Authorization": f"Bearer {DEEPSEEK_API_KEY}", "Content-Type": "application/json"}
    payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": "Return ONLY the updated prompt text. No markdown. No JSON."},
            {"role": "user", "content": meta_prompt},
        ],
        "temperature": 0.0,
    }

    t0 = time.perf_counter()
    resp = requests.post(url, headers=headers, json=payload, timeout=180)
    elapsed = time.perf_counter() - t0
    resp.raise_for_status()
    data = resp.json()
    # 2 😇
    print('Did recieve response from deepseek 2')
    # ---- Stop reason (transport-level) ----
    choice0 = (data.get("choices") or [{}])[0]
    finish_reason = choice0.get("finish_reason", "")  # e.g. "stop", "length", ...
    usage = data.get("usage", {}) or {}

    # Heuristic: "length" usually means context/output limit.
    if finish_reason == "length":
        emoji = "🔴"
        reason_human = "Stopped due to token/context limit (finish_reason=length)."
    elif finish_reason in ("stop", "eos"):
        emoji = "🟢"
        reason_human = f"Completed normally (finish_reason={finish_reason})."
    else:
        emoji = "🟡"
        reason_human = f"Stopped for other reason (finish_reason={finish_reason})."

    print(f"LLM latency: {elapsed:.2f}s")
    print(f"{emoji} {reason_human}")
    if usage:
        # Some providers return prompt_tokens/completion_tokens/total_tokens
        print("Token usage:", json.dumps(usage, ensure_ascii=False))

    new_prompt = choice0.get("message", {}).get("content", "")
    if not new_prompt.strip():
        raise RuntimeError("Empty prompt returned by LLM.")

    with open(PROMPT_PATH, "w", encoding="utf-8") as f:
        f.write(new_prompt)

    print("✅ Updated prompt saved to prompt.txt")
    return new_prompt

In [173]:
# 5 JSON → Cypher генерация

In [174]:
from typing import List
def step_4_abox_to_cypher(LLM_answer: str) -> List[str]:
    import json
    import re
    from pathlib import Path
    from typing import List
    # =========================
    # Helpers
    # =========================

    def strip_code_fences(s: str) -> str:
        s = (s or "").strip()
        if s.startswith("```"):
            s = re.sub(r"^```(?:json)?\s*", "", s)
            s = re.sub(r"\s*```$", "", s)
        return s.strip()

    def neo4j_local_name(name: str) -> str:
        if not name:
            return ""
        if ":" in name:
            return name.split(":")[-1]
        if "#" in name:
            return name.split("#")[-1]
        if "/" in name:
            return name.rsplit("/", 1)[-1]
        return name

    def neo4j_safe_ident(name: str) -> str:
        n = neo4j_local_name(name)
        n = re.sub(r"[^A-Za-z0-9_]", "_", n)
        if not n:
            return "X"
        if n[0].isdigit():
            n = "_" + n
        return n

    def cypher_literal(v, prop_name=None):
        if v is None:
            return "null"
        if isinstance(v, bool):
            return "true" if v else "false"
        if isinstance(v, (int, float)):
            return str(v)

        # Special handling for ISO date → Neo4j localdatetime()
        if prop_name == "postDateISO":
            return f"localdatetime('{v}')"

        s = str(v)
        s = s.replace("\\", "\\\\").replace("'", "\\'")
        s = s.replace("\n", "\\n").replace("\r", "\\r")
        return f"'{s}'"

    def contains_cyrillic(s):
        return bool(re.search(r"[А-Яа-яЁё]", s))

    # =========================
    # Parse JSON from LLM
    # =========================

    # global answer  # must exist
    answer = LLM_answer
    raw = strip_code_fences(answer)
    root = json.loads(raw)

    payload = root.get("payload", {}) or {}
    nodes = payload.get("nodes", []) or []
    rels = payload.get("relationships", []) or []

    cypher_statements = []

    # =========================
    # Constraints
    # =========================

    labels = sorted({
        neo4j_safe_ident(n.get("label", ""))
        for n in nodes
        if n.get("label")
    })

    for label in labels:
        cypher_statements.append(
            f"CREATE CONSTRAINT IF NOT EXISTS FOR (n:{label}) REQUIRE n.key IS UNIQUE"
        )

    # =========================
    # Nodes
    # =========================

    for node in nodes:
        label_raw = node.get("label", "")
        key = node.get("key")

        if not label_raw or key is None:
            continue

        label = neo4j_safe_ident(label_raw)
        props = node.get("properties", {}) or {}

        set_parts = []

        for k, v in props.items():
            prop_key = neo4j_safe_ident(k)

            if isinstance(v, str) and contains_cyrillic(v):
                print(f"⚠ WARNING: Cyrillic detected in property '{prop_key}' → '{v}'")

            set_parts.append(f"n.{prop_key}={cypher_literal(v, prop_key)}")

        set_clause = (" SET " + ", ".join(set_parts)) if set_parts else ""

        stmt = f"MERGE (n:{label} {{key:{cypher_literal(key)}}}){set_clause}"
        cypher_statements.append(stmt)

    # =========================
    # Relationships
    # =========================

    for rel in rels:
        f = rel.get("from", {}) or {}
        t = rel.get("to", {}) or {}

        fl_raw = f.get("label", "")
        fk = f.get("key")
        tl_raw = t.get("label", "")
        tk = t.get("key")
        rt_raw = rel.get("type", "")

        if not fl_raw or not tl_raw or not rt_raw or fk is None or tk is None:
            continue

        fl = neo4j_safe_ident(fl_raw)
        tl = neo4j_safe_ident(tl_raw)
        rt = neo4j_safe_ident(rt_raw)

        stmt = (
            f"MATCH (a:{fl} {{key:{cypher_literal(fk)}}}), "
            f"(b:{tl} {{key:{cypher_literal(tk)}}}) "
            f"MERGE (a)-[:{rt}]->(b)"
        )

        cypher_statements.append(stmt)

    # =========================
    # Output
    # =========================

    print("LLM finish_reason:", root.get("finish_reason"))
    print("LLM reason:", root.get("reason"))
    print(f"Prepared {len(cypher_statements)} Cypher statements.\n")
    # cypher_statements = abox_json_to_cypher(answer)
    # for s in cypher_statements[:15]:
    #     print(s + ";")
    return cypher_statements
    


In [175]:
# 6 Сохранение Cypher в чекпоинт

In [176]:
def step_5_save_cypher(cypher_statements: List[str]):
    import os
    import re
    
    ckpt_dir = "cypher_checkpoints"
    os.makedirs(ckpt_dir, exist_ok=True)
    
    existing = []
    for name in os.listdir(ckpt_dir):
        m = re.match(r"^cypher_(\d+)\.txt$", name)
        if m:
            existing.append(int(m.group(1)))
    
    next_num = max(existing, default=0) + 1
    out_path = os.path.join(ckpt_dir, f"cypher_{next_num}.txt")
    
    with open(out_path, "w", encoding="utf-8") as f:
        for stmt in cypher_statements:
            f.write(stmt)
            f.write(";")
    
    print(f"Saved {len(cypher_statements)} statements to {out_path}")
    return None


In [177]:
# print(cypher_statements)

In [178]:
# 7 Запись в базу. исполнение Cypher (каждая строка автономна)

In [179]:
def step_6_execute_cypher(cypher_statements: List[str]):
    from neo4j import GraphDatabase
    
    URI = "neo4j://localhost:7688"
    USER = "neo4j"
    PASSWORD = "12345678"
    DATABASE = "neo4j"
    
    driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))
    
    try:
        with driver.session(database=DATABASE) as session:
            for i, stmt in enumerate(cypher_statements, 1):
                session.run(stmt).consume()
        print(f"✅ Executed {len(cypher_statements)} statements.")
    finally:
        driver.close()    
    return None


In [158]:
# 8 запуск пайплайна

In [180]:
step_1_reset_db()

import json
import subprocess

with open("messages.json", "r", encoding="utf-8") as f:
    messages = json.load(f)
messages = list(map(lambda x: json.dumps(x, ensure_ascii=False), messages[:1]))


for msg in messages:
    print(msg)
    abox_json = step_2_prompt_to_abox(msg) # первое извлеч LLM_1
    improve_prompt_with_llm(msg, abox_json) # испр промпта LLM_2
    abox_json = step_2_prompt_to_abox(msg) # второе извлеч LLM_3
    cypher_statements = step_4_abox_to_cypher(abox_json)
    step_5_save_cypher(cypher_statements)
    step_6_execute_cypher(cypher_statements)

subprocess.run(["afplay", "/System/Library/Sounds/Blow.aiff"], check=False)


neo4j_travel
neo4j_travel
neo4j_travel_data
1dc1c0838e424f93cd43f760cbf4a7a2bd795c408e9c7385ea707cf8e41bfe8b
Did reset DB
{"post_id": 3902451, "post_url": "https://forum.awd.ru/viewtopic.php?p=3902451#p3902451", "author": "!Casper!", "date": "11 июн 2013, 09:05", "text": "Прилетали вдвоем с другом в Париж, ШДГ, друг прошел контроль за 10 сек максиму без каких либо вопросов, а вот мне устроили допрос минут на 5. Попросили показать обратный билет, показать страховку, бронь отеля, спросили оплачен ли отель, показать кредитные карты, наличные деньги, спросили сколько наличности при себе, спросили какой картой я оплачивал отель, цель визита, на сколько дней приехал, кем я работаю в России. После этого поставили штамп в паспорт! Вывод: держите все распечатки билетов, броней отеля, страховки и тд при себе, так как могут возникнуть проблемы!"}
DeepSeek API latency: 208.85s
Did recieve response from deepseek 1
🟢 API finish_reason = stop
{
  "payload": {
    "ontology_version": "pcfr_v3",
    "n

CompletedProcess(args=['afplay', '/System/Library/Sounds/Blow.aiff'], returncode=0)

In [ ]:
было 93 104 97 113 для ШДГ и 25 23 для малого 

In [ ]:
# 8 RAG: LLM -> Cypher -> facts -> LLM


In [10]:
def rag_answer(user_query: str) -> None:
    from neo4j import GraphDatabase
    import os
    import requests
    import json

    # -----------------------------
    # Config
    # -----------------------------
    URI = "neo4j://localhost:7688"
    USER = "neo4j"
    PASSWORD = "12345678"
    DATABASE = "neo4j"

    DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
    if not DEEPSEEK_API_KEY:
        raise RuntimeError("Set env var DEEPSEEK_API_KEY first.")

    DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL", "https://api.deepseek.com").rstrip("/")
    MODEL = os.getenv("DEEPSEEK_MODEL", "deepseek-reasoner")

    url = f"{DEEPSEEK_BASE_URL}/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {DEEPSEEK_API_KEY}",
        "Content-Type": "application/json",
    }

    # -----------------------------
    # Helper: call DeepSeek
    # -----------------------------
    def call_llm(system_prompt: str, user_prompt: str, temperature: float = 0.0) -> str:
        payload = {
            "model": MODEL,
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            "temperature": temperature,
        }

        resp = requests.post(url, headers=headers, json=payload, timeout=180)
        resp.raise_for_status()
        data = resp.json()
        return data["choices"][0]["message"]["content"]

    # =========================================================
    # 0️⃣ Extract DB schema (real schema injection)
    # =========================================================

    driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))

    try:
        with driver.session(database=DATABASE) as session:

            labels = session.run(
                "CALL db.labels() YIELD label RETURN collect(label) AS labels"
            ).single()["labels"]

            rel_types = session.run(
                "CALL db.relationshipTypes() YIELD relationshipType "
                "RETURN collect(relationshipType) AS relTypes"
            ).single()["relTypes"]

            label_props = {}
            for label in labels:
                result = session.run(
                    f"MATCH (n:{label}) RETURN keys(n) AS keys LIMIT 1"
                ).single()
                label_props[label] = result["keys"] if result else []

    finally:
        driver.close()

    schema_text = (
        f"Available node labels: {labels}\n"
        f"Available relationship types: {rel_types}\n"
        f"Node properties by label: {json.dumps(label_props)}\n"
    )

    # =========================================================
    # 1️⃣ Generate Cypher (STRICT schema-aware prompt)
    # =========================================================

    system_cypher = """
You are a Neo4j Cypher generator.

INPUT YOU WILL RECEIVE:
- Database schema text (labels, relationship types, property keys).
- A user question.

HARD RULES (MUST FOLLOW):
1) Use ONLY the labels, relationship types, and property keys present in the provided schema text.
2) Do NOT invent any labels, relationship types, or properties.
3) Read-only queries only: MATCH / OPTIONAL MATCH / WHERE / WITH / RETURN.
   - Do NOT use CREATE, MERGE, DELETE, SET, CALL.
4) Always include LIMIT.
   - Use LIMIT 50 for list outputs.
   - Use LIMIT 1 for aggregated outputs.
5) If filtering by real-world entities (France, Paris, CDG),
   use entityName or key only if visible in schema sample.
   Never guess key format.
6) Be robust:
   - Use OPTIONAL MATCH when appropriate.
   - Use DISTINCT to avoid double counting.

FREQUENCY RULE (CRITICAL):
If the question asks about frequency / how often / percentage:
- Compute:
    total_count   = total relevant Encounter in scope
    match_count   = number satisfying condition
    share         = toFloat(match_count) / total_count
- Return ALL THREE fields.

OUTPUT FORMAT:
Return ONLY valid JSON:
{"cypher": "<ONE Cypher query>"}
No markdown. No explanation.
"""

    prompt_cypher = (
        f"Database schema:\n{schema_text}\n\n"
        f"User question:\n{user_query}\n\n"
        "Generate Cypher query."
    )

    cypher_json = call_llm(system_cypher, prompt_cypher, temperature=0.0)

    try:
        cypher_obj = json.loads(cypher_json)
        cypher = cypher_obj.get("cypher", "").strip()
    except Exception:
        cypher = ""

    # =========================================================
    # 2️⃣ Execute Cypher
    # =========================================================

    facts = []

    if cypher:
        driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))
        try:
            with driver.session(database=DATABASE) as session:
                records = session.run(cypher).data()
                facts.extend(records)
        finally:
            driver.close()

    facts_text = json.dumps(facts, ensure_ascii=False)

    # =========================================================
    # 3️⃣ Generate Final Answer
    # =========================================================

    system_answer = (
        "You are a concise assistant.\n"
        "If the query result contains total_count, match_count and share, "
        "interpret them as frequency statistics.\n"
        "Use ONLY the provided database facts.\n"
        "If no relevant data exists, say you do not know."
    )

    prompt_answer = (
        f"Question:\n{user_query}\n\n"
        f"Facts:\n{facts_text}\n\n"
        "Answer:"
    )

    final_answer = call_llm(system_answer, prompt_answer, temperature=0.2)

    print("\n--- Injected Schema ---")
    print(schema_text)

    print("\n--- Generated Cypher ---")
    print(cypher)

    print("\n--- Retrieved Facts ---")
    print(facts_text)

    print("\n--- Final Answer ---")
    print(final_answer)

In [4]:
# def rag_answer(user_query: str) -> None:
#     from neo4j import GraphDatabase
#     import os
#     import requests

#     URI = "neo4j://localhost:7688"
#     USER = "neo4j"
#     PASSWORD = "12345678"
#     DATABASE = "neo4j"

#     driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))

#     try:
#         with driver.session(database=DATABASE) as session:
#             cypher = """
#             MATCH (n)
#             WHERE any(k IN keys(n) WHERE toString(n[k]) CONTAINS $q)
#             RETURN n
#             LIMIT 20
#             """
#             records = session.run(cypher, q=user_query).data()
#     finally:
#         driver.close()

#     # Build minimal context
#     context_lines = []
#     for r in records:
#         n = r.get("n")
#         if n is None:
#             continue
#         labels = list(n.labels) if hasattr(n, "labels") else []
#         props = dict(n)
#         context_lines.append(f"labels={labels}; props={props}")

#     context = "\n".join(context_lines) if context_lines else "NO_MATCHES"

#     DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
#     if not DEEPSEEK_API_KEY:
#         raise RuntimeError("Set env var DEEPSEEK_API_KEY first.")

#     DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL", "https://api.deepseek.com").rstrip("/")
#     MODEL = os.getenv("DEEPSEEK_MODEL", "deepseek-chat")

#     prompt = (
#         "You are a concise assistant. Use ONLY the provided context to answer. "
#         "If the answer is not in context, say you do not know.\n\n"
#         f"Context:\n{context}\n\n"
#         f"Question: {user_query}\n"
#         "Answer:"
#     )

#     url = f"{DEEPSEEK_BASE_URL}/v1/chat/completions"
#     headers = {
#         "Authorization": f"Bearer {DEEPSEEK_API_KEY}",
#         "Content-Type": "application/json",
#     }

#     payload = {
#         "model": MODEL,
#         "messages": [
#             {"role": "system", "content": "Answer succinctly and only from context."},
#             {"role": "user", "content": prompt},
#         ],
#         "temperature": 0.2,
#     }

#     resp = requests.post(url, headers=headers, json=payload, timeout=180)
#     resp.raise_for_status()
#     data = resp.json()

#     content = data["choices"][0]["message"]["content"]
#     print(content)

In [8]:
# 9 Проверка RAG

In [11]:
import subprocess
# rag_answer('Является ли паспортный контроль в Париже строгим?')
# rag_answer('Is passport control in Paris strict?')
# rag_answer('Is passport control in Paris takes much time?')
rag_answer('How often is return ticket required on passport control in France?')
subprocess.run(["afplay", "/System/Library/Sounds/Blow.aiff"], check=False)


Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The semantics of using colon in the separation of alternative relationship types will change in a future version. (Please use ':atCountry|aboutCountry' instead)} {position: line: 1, column: 32, offset: 31} for query: "MATCH (e:Encounter)-[:atCountry|:aboutCountry]->(c:Country) WHERE c.entityName = 'France' WITH DISTINCT e OPTIONAL MATCH (e)-[:hasDocumentCheck]->(dc:DocumentCheck)-[:requestedDocType]->(rt:ReturnTicket), (dc)-[:treatedAsRequirementLevel]->(r:Required) WITH e, CASE WHEN rt IS NOT NULL AND r IS NOT NULL THEN 1 ELSE 0 END as is_required WITH e, MAX(is_required) as has_required_return WITH COUNT(e) as total_count, SUM(has_required_return) as match_count RETURN total_count, match_count, toFloat(match_count) / total_count as share LIMIT 1"



--- Injected Schema ---
Available node labels: ['Advice', 'Airport', 'Cash', 'CashAmount', 'City', 'Claim', 'Country', 'DocumentCheck', 'DocumentScrutiny', 'Employment', 'Encounter', 'Experienced', 'FirstPersonExperience', 'ForeignPassport', 'HotelBookingDoc', 'HotelPaymentCard', 'HotelPaymentStatus', 'Outcome', 'PaymentCard', 'Questioning', 'ReturnTicket', 'SecondaryInterrogation', 'Stamping', 'TravelInsurance', 'Traveller', 'UniversalScope', 'VisitDuration', 'VisitPurpose', 'GeneralKnowledge', 'Generic', 'HabitualScope', 'ProcessingSpeed', 'StampingPractice', 'VerbalInquiryOnly', 'BorderOfficer', 'Printout', 'PhysicalDocumentRequested', 'Required', 'Heard', 'SingleEventScope', 'NoExtraProcedures', 'PassportWithVisa', 'Inferred']
Available relationship types: ['locatedInCity', 'locatedInCountry', 'atAirport', 'atCity', 'atCountry', 'hasTraveler', 'hasScreeningStatus', 'assertionKind', 'evidentiality', 'hasQuestioning', 'hasStamping', 'hasOutcome', 'hasDocumentCheck', 'requestedDocTyp

CompletedProcess(args=['afplay', '/System/Library/Sounds/Blow.aiff'], returncode=0)

In [ ]:
--- Final Answer ---
Based on the provided frequency statistics, a return ticket is required in approximately 14.29% of passport control instances in France, which corresponds to 2 out of 14 cases.

In [ ]:
# ??? Запись в базу уже из файла с диска 

In [120]:
from __future__ import annotations
from pathlib import Path
from typing import Optional, Iterable
from neo4j import GraphDatabase


def _split_cypher_statements(text: str) -> list[str]:
    """
    Split on semicolons, ignoring empty chunks.
    (Assumes your generator doesn't put ';' inside string literals.)
    """
    parts = [p.strip() for p in text.split(";")]
    return [p for p in parts if p]


def run_cypher_file(
    cypher_path: str,
    uri: str = "neo4j://localhost:7688",
    user: str = "neo4j",
    password: str = "12345678",
    database: Optional[str] = None,
) -> None:
    """
    Reads Cypher statements from a file and executes them sequentially.
    Each statement is run as its own query (Neo4j requires 1 statement per run()).
    """
    path = Path(cypher_path)
    if not path.exists():
        raise FileNotFoundError(f"Cypher file not found: {path}")

    cypher_text = path.read_text(encoding="utf-8")
    statements = _split_cypher_statements(cypher_text)

    if not statements:
        print("No Cypher statements found in file.")
        return

    driver = GraphDatabase.driver(uri, auth=(user, password))
    try:
        with driver.session(database=database) as session:
            for i, stmt in enumerate(statements, start=1):
                session.run(stmt).consume()  # consume forces execution now
                # print(f"✅ {i}/{len(statements)} executed")
        print(f"✅ Executed {len(statements)} statements from {path.name}")
    finally:
        driver.close()

run_cypher_file("cypher_checkpoints/cypher_1.txt", uri="neo4j://localhost:7688", user="neo4j", password="12345678")

✅ Executed 113 statements from cypher_1.txt
